# Joblib — Model Serialization

In [ ]:
# Day 30 — Joblib & Model Serialization
# After training a model, you need to SAVE it to disk
# So you can LOAD it later without retraining
#
# Analogy:
# Training a model = cooking a meal (takes time and ingredients)
# Saving the model = putting leftovers in the fridge
# Loading the model = heating up leftovers (instant, no cooking needed)
#
# Today we'll cover:
#   1. Why save models (don't retrain every time!)
#   2. pickle — Python's built-in serializer
#   3. joblib — better for ML models (handles large NumPy arrays)
#   4. Saving & loading sklearn models
#   5. Saving & loading pipelines
#   6. Versioning saved models
#   7. Best practices for model artifacts
#
# Install: joblib comes with scikit-learn

### Why Save Models?

In [ ]:
# Without saving:
#   Train model (takes 30 min) → Use it → Close Python → MODEL IS GONE!
#   Next day: train again (30 min wasted)
#
# With saving:
#   Train model → Save to file → Close Python
#   Next day: Load from file (instant!) → Use it
#
# In MLOps, the workflow is:
# 1. Data scientist trains and saves the best model
# 2. DevOps deploys the saved model to production (FastAPI, Docker, K8s)
# 3. Production server LOADS the saved model and serves predictions
#
# The saved model file is called a MODEL ARTIFACT

# Let's first train a model to save

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

np.random.seed(42)
n = 300
X = pd.DataFrame({
    'cpu': np.random.randint(10, 100, n),
    'memory': np.random.randint(15, 95, n),
    'disk': np.random.randint(10, 90, n),
    'error_count': np.random.randint(0, 50, n)
})
score = 0.3*X['cpu'] + 0.25*X['memory'] + 0.15*X['disk'] + 0.8*X['error_count'] + np.random.normal(0, 10, n)
y = (score > 50).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])
pipe.fit(X_train, y_train)
print(f'Model trained! Accuracy: {pipe.score(X_test, y_test):.2%}')

### pickle — Python's Built-in Serializer

In [ ]:
# pickle converts Python objects to bytes and back
# It can save ANY Python object: lists, dicts, models, etc.

import pickle

# SAVE the model
with open('model_pickle.pkl', 'wb') as f:    # 'wb' = write binary
    pickle.dump(pipe, f)
print('Model saved with pickle!')

# LOAD the model
with open('model_pickle.pkl', 'rb') as f:    # 'rb' = read binary
    loaded_model = pickle.load(f)

# Use the loaded model
print(f'Loaded model accuracy: {loaded_model.score(X_test, y_test):.2%}')

# Predict with loaded model
new_server = [[85, 90, 70, 25]]
pred = loaded_model.predict(new_server)
print(f'Prediction: {"CRITICAL" if pred[0] == 1 else "HEALTHY"}')

### joblib — Better for ML Models

In [ ]:
# joblib is PREFERRED over pickle for ML models because:
# 1. More efficient with large NumPy arrays (common in ML)
# 2. Built-in compression support
# 3. Faster for sklearn models specifically
# 4. sklearn documentation recommends it

import joblib

# SAVE the model
joblib.dump(pipe, 'model_joblib.pkl')
print('Model saved with joblib!')

# SAVE with compression (smaller file size)
joblib.dump(pipe, 'model_compressed.pkl.gz', compress=3)
print('Compressed model saved!')

# LOAD the model
loaded_model = joblib.load('model_joblib.pkl')
print(f'Loaded model accuracy: {loaded_model.score(X_test, y_test):.2%}')

# Compare file sizes
import os
print(f'\nPickle size:     {os.path.getsize("model_pickle.pkl") / 1024:.1f} KB')
print(f'Joblib size:     {os.path.getsize("model_joblib.pkl") / 1024:.1f} KB')
print(f'Compressed size: {os.path.getsize("model_compressed.pkl.gz") / 1024:.1f} KB')

### Saving & Loading Individual Components

In [ ]:
# Sometimes you want to save the scaler and model SEPARATELY
# This gives you more control in deployment

import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Train components separately
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Save EACH component
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(model, 'model.pkl')
print('Scaler and model saved separately!')

# Load and use them
loaded_scaler = joblib.load('scaler.pkl')
loaded_model = joblib.load('model.pkl')

# Predict with loaded components
new_data = [[85, 90, 70, 25]]
new_data_scaled = loaded_scaler.transform(new_data)    # scale first
prediction = loaded_model.predict(new_data_scaled)     # then predict
print(f'Prediction: {"CRITICAL" if prediction[0] == 1 else "HEALTHY"}')

### Saving Model Metadata

In [ ]:
# Best practice: save MODEL + METADATA together
# Metadata tells you EVERYTHING about the model:
# what it does, how it was trained, how good it is

import joblib
import json
from datetime import datetime
from sklearn.metrics import accuracy_score, f1_score

# Calculate metrics
y_pred = pipe.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Create metadata
metadata = {
    'model_name': 'server_health_classifier',
    'version': 'v1.0.0',
    'model_type': 'RandomForestClassifier',
    'features': ['cpu', 'memory', 'disk', 'error_count'],
    'target': 'health_status (0=healthy, 1=critical)',
    'metrics': {
        'accuracy': round(accuracy, 4),
        'f1_score': round(f1, 4)
    },
    'training_samples': len(X_train),
    'test_samples': len(X_test),
    'trained_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model_file': 'server_health_v1.0.0.pkl',
    'notes': 'Pipeline includes StandardScaler + RandomForest(n_estimators=100)'
}

# Save model
joblib.dump(pipe, metadata['model_file'])

# Save metadata
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print('Model + metadata saved!')
print(json.dumps(metadata, indent=4))

In [ ]:
# Loading in production

import joblib
import json

# Load metadata first to know what we're dealing with
with open('model_metadata.json', 'r') as f:
    meta = json.load(f)

print(f'Loading model: {meta["model_name"]} {meta["version"]}')
print(f'Features: {meta["features"]}')
print(f'Accuracy: {meta["metrics"]["accuracy"]}')

# Load the model
model = joblib.load(meta['model_file'])

# Use in production
import numpy as np
incoming_data = np.array([[75, 85, 60, 30]])
result = model.predict(incoming_data)
print(f'\nPrediction: {"CRITICAL" if result[0] == 1 else "HEALTHY"}')

In [ ]:
# Practice Questions:

# 1. Train a model, save it with joblib, close and reopen the notebook,
#    load the model, and make predictions (verify it works across sessions)

# 2. Save the same model with pickle and joblib
#    Compare: file sizes, load times (use time.time())

# 3. Write a function save_model(model, name, version, metrics_dict) that:
#    - Saves the model as {name}_v{version}.pkl
#    - Creates a metadata JSON file
#    - Returns the paths of both files

# 4. Write a function load_latest_model(model_name) that:
#    - Reads metadata JSON to find the model file
#    - Loads and returns the model
#    - Prints model info (version, accuracy, features)

### DevOps / AI-MLOps Practice

In [ ]:
# Model serialization is the BRIDGE between training and deployment

# Exercise 1: Model Artifact Manager
# Create a class ModelArtifact that:
#   __init__(self, name, version)
#   save(self, model, metrics, features) -> saves .pkl + .json
#   load(self) -> loads model from .pkl, returns model
#   info(self) -> prints metadata from .json
# Test it with a trained model

# Exercise 2: Model Versioning
# Train 3 different versions of a model (different hyperparams)
# Save each as v1.0, v1.1, v1.2 with metadata
# Write a function that loads ALL versions, compares accuracy,
# and returns the best one
# This is a simplified version of what MLflow does (Day 34)!

# Exercise 3: Deployment-Ready Package
# Create a folder structure:
#   model_package/
#     model.pkl
#     metadata.json
#     requirements.txt  (list the pip packages needed)
# Write a predict.py script that:
#   - Loads the model
#   - Accepts input from command line or function call
#   - Returns predictions
# This is what you'd put in a Docker container for deployment!